# Tải Dataset và Output Notebook từ Kaggle về Google Drive
Notebook này giúp bạn tự động cấu hình `Kaggle API Token`, kết nối với Google Drive và tải kết quả (output) từ các notebook Kaggle bạn chọn về thư mục `MyDrive/AIC 2026`.

In [ ]:
import os
import getpass
from google.colab import drive

# 1. Kết nối với Google Drive
drive.mount('/content/drive')

# 2. Tạo thư mục đích trên Drive
target_dir = '/content/drive/MyDrive/AIC 2026'
os.makedirs(target_dir, exist_ok=True)
print(f"\nThư mục lưu trữ đã sẵn sàng: {target_dir}")

# 3. Cấu hình môi trường Kaggle (Sử dụng API Token mới)
print("\n=== VUI LÒNG NHẬP KAGGLE API TOKEN ===")
print("(Token có dạng: KGAT_...)")
api_token = getpass.getpass("Nhập Kaggle API Token của bạn: ")
os.environ['KAGGLE_API_TOKEN'] = api_token.strip()

print("\nCấu hình Kaggle API thành công!")

## Tải Output từ các Kaggle Notebook
Bạn có thể lấy output data từ bất kỳ Kaggle Notebook nào bằng cách dán đường dẫn của nó (ví dụ: `https://www.kaggle.com/code/username/notebook-name`).

In [ ]:
import subprocess

# Nhập danh sách các link notebook, cách nhau bằng dấu phẩy
notebook_urls = input("Nhập các đường dẫn Kaggle Notebook (cách nhau bởi dấu phẩy): ")

urls = [url.strip() for url in notebook_urls.split(",") if url.strip()]

print(f"\nBắt đầu tải {len(urls)} notebooks về thư mục: {target_dir}\n")
print("=" * 50)

for url in urls:
    # Xóa bỏ các tham số phụ đằng sau dấu '?' nếu có trong URL
    clean_url = url.split('?')[0]
    
    # Tách chuỗi URL để lấy kernel slug (username/kernel-name)
    parts = clean_url.split('/')
    try:
        if 'code' in parts:
            idx = parts.index('code')
            slug = f"{parts[idx+1]}/{parts[idx+2]}"
        else:
            slug = f"{parts[-2]}/{parts[-1]}"
            
        print(f"Đang tải output từ: {slug}...")
        
        # Chạy lệnh Kaggle API để tải output của kernel
        cmd = f"kaggle kernels output {slug} -p '{target_dir}'"
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        
        if result.returncode == 0:
            print(f"✅ Hoàn tất tải: {slug}")
            print(result.stdout)
        else:
            print(f"❌ Lỗi khi tải {slug}")
            print(result.stderr)
            
    except Exception as e:
        print(f"❌ Lỗi khi phân tích URL {url}: {e}")
    
    print("=" * 50)

print("\nTất cả tiến trình đã hoàn thành!")

## Tải Dataset từ Kaggle (Cách 1: Lưu nguyên vẹn file ZIP lên Drive)
Đây là phương pháp TIÊU CHUẨN VÀNG cho AI/Deep Learning:
- Lưu thẳng file `.zip` của Dataset vào Drive (thư mục `AIC 2026/Dataset_Zips`).
- Không giải nén để tránh lỗi rớt file, tiết kiệm I/O của Drive và tải siêu nhanh.
- Lúc cần Train Model, chỉ cần 1 lệnh bung nén trực tiếp vào `/content`.

In [ ]:
import os
import shutil

dataset_urls = input("Nhập đường dẫn Kaggle Dataset (cách nhau bởi dấu phẩy): ")
d_urls = [url.strip() for url in dataset_urls.split(",") if url.strip()]
temp_dir = '/content/temp_dataset'

# Thư mục mới để chứa toàn bộ các file Zip trên Drive
zip_target_dir = os.path.join(target_dir, 'Dataset_Zips')
os.makedirs(zip_target_dir, exist_ok=True)

for url in d_urls:
    clean_url = url.split('?')[0]
    parts = clean_url.split('/')
    try:
        if 'datasets' in parts:
            idx = parts.index('datasets')
            slug = f"{parts[idx+1]}/{parts[idx+2]}"
        else:
            slug = f"{parts[-2]}/{parts[-1]}"
            
        print(f"\n======================================")
        print(f"BẮT ĐẦU TẢI DATASET: {slug}")
        print(f"======================================\n")
        
        # 1. Dọn dẹp và tải file zip về thư mục tạm trên bộ nhớ siêu tốc SSD của Colab
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)
        os.makedirs(temp_dir, exist_ok=True)
        
        print("Đang tải file từ Kaggle (Vui lòng theo dõi thanh tiến trình bên dưới):")
        !kaggle datasets download -d {slug} -p '{temp_dir}'
        
        # 2. Kiểm tra file zip (Xử lý mượt lỗi 404 Private/Rỗng)
        zip_files = [f for f in os.listdir(temp_dir) if f.endswith('.zip')]
        if len(zip_files) == 0:
            print("\n❌ LỖI NGHIÊM TRỌNG: Không tải được file zip nào về máy!")
            print("👉 Nguyên nhân 1: Dataset trên Kaggle đang bị ĐẶT PRIVATE.")
            print("👉 Nguyên nhân 2: Dataset chưa có file nào (0 bytes rỗng tuếch).")
            continue
            
        zip_file = zip_files[0]
        zip_path = os.path.join(temp_dir, zip_file)
        target_zip_path = os.path.join(zip_target_dir, zip_file)
        
        # 3. Đẩy nguyên cục ZIP sang Google Drive
        if os.path.exists(target_zip_path):
            print(f"\n ⏭ Bỏ qua file: {zip_file} (Vì đã có sẵn trên Drive rồi)")
        else:
            print(f"\n➔ Đang đẩy nguyên vẹn khối {zip_file} sang Google Drive...")
            print("   (Tốc độ lưu cực nhanh, không lo rớt file)")
            shutil.move(zip_path, target_zip_path)
            print(f"✅ Đã lưu trữ an toàn tại: {target_zip_path}")
            
        # 4. Dọn dẹp rác
        shutil.rmtree(temp_dir)
        print(f"\n✅ HOÀN TẤT TOÀN BỘ QUY TRÌNH CHO {slug}!")
        
    except Exception as e:
        print(f"❌ Lỗi Hệ Thống: {e}")

### 💡 Hướng dẫn sử dụng file ZIP lúc viết code Train AI:
Khi nào bạn mở một file Colab mới để bắt đầu train model, bạn chỉ cần dùng ô code đơn giản dưới đây để bung 100,000 ảnh từ file Zip vào ổ cứng SSD siêu tốc của Colab. Tốc độ Train sẽ nhanh gấp 100 lần so với đọc trực tiếp từ Drive:

```python
import os
from google.colab import drive
drive.mount('/content/drive')

# Lệnh bung file Zip từ Drive vào trực tiếp máy ảo (Rất nhanh)
!unzip -q '/content/drive/MyDrive/AIC 2026/Dataset_Zips/dataset-part-2.zip' -d '/content/Data_Train'
!unzip -q '/content/drive/MyDrive/AIC 2026/Dataset_Zips/dataset-part-3.zip' -d '/content/Data_Train'

# Khai báo đường dẫn data cho code AI đọc
DATA_PATH = '/content/Data_Train/Keyframes'
```